In [2]:
import sys
# Remove the repo root from sys.path so Python imports the installed package
# instead of the local uncompiled source
sys.path = [p for p in sys.path if p not in ("", ".", "d:\\nautilus_trader", "d:/nautilus_trader", "D:\\nautilus_trader", "D:/nautilus_trader")]
from decimal import Decimal

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.examples.algorithms.twap import TWAPExecAlgorithm
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAP
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAPConfig
from nautilus_trader.model import BarType
from nautilus_trader.model import Money
from nautilus_trader.model import TraderId
from nautilus_trader.model import Venue
from nautilus_trader.model.currencies import ETH           
from nautilus_trader.model.currencies import USDT
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.wranglers import TradeTickDataWrangler
from nautilus_trader.test_kit.providers import TestDataProvider
from nautilus_trader.test_kit.providers import TestInstrumentProvider

In [4]:
import pandas as pd
from nautilus_trader.persistence.wranglers import TradeTickDataWrangler
from nautilus_trader.test_kit.providers import TestInstrumentProvider

# Load CSV (FIX PATH)
trades_df = pd.read_csv(r"D:\nautilus_trader\nautilus_trader\tests\test_data\binance\ethusdt-trades.csv")

# 🔧 Convert timestamp → datetime (VERY IMPORTANT)
trades_df["timestamp"] = pd.to_datetime(
    trades_df["timestamp"],
    unit="ns",      # nanoseconds (your sample format)
    utc=True
)

# 🔧 Set timestamp as index (REQUIRED by Nautilus)
trades_df = trades_df.set_index("timestamp")

# Define instrument
ETHUSDT_BINANCE = TestInstrumentProvider.ethusdt_binance()

# Process data into Nautilus ticks
wrangler = TradeTickDataWrangler(instrument=ETHUSDT_BINANCE)
ticks = wrangler.process(trades_df)

# Preview
ticks[:5]

[TradeTick(ETHUSDT.BINANCE,423.76,2.67900,SELLER,148568980,1597399200223000000),
 TradeTick(ETHUSDT.BINANCE,423.74,2.31976,SELLER,148568981,1597399200976000000),
 TradeTick(ETHUSDT.BINANCE,423.73,2.16924,SELLER,148568982,1597399200976000000),
 TradeTick(ETHUSDT.BINANCE,423.68,0.19096,SELLER,148568983,1597399201185000000),
 TradeTick(ETHUSDT.BINANCE,423.70,0.82490,BUYER,148568984,1597399201913000000)]

In [5]:
# Configure backtest engine
config = BacktestEngineConfig(trader_id=TraderId("BACKTESTER-001"))

# Build the backtest engine
engine = BacktestEngine(config=config)

In [6]:
# Add a trading venue (multiple venues possible)
BINANCE = Venue("BINANCE")
engine.add_venue(
    venue=BINANCE,
    oms_type=OmsType.NETTING,
    account_type=AccountType.CASH,  # Spot CASH account (not for perpetuals or futures)
    base_currency=None,  # Multi-currency account
    starting_balances=[Money(1_000_000.0, USDT), Money(10.0, ETH)],
)

In [7]:
# Add instrument(s)
engine.add_instrument(ETHUSDT_BINANCE)

# Add data
engine.add_data(ticks)

In [8]:
# Configure your strategy
strategy_config = EMACrossTWAPConfig(
    instrument_id=ETHUSDT_BINANCE.id,
    bar_type=BarType.from_str("ETHUSDT.BINANCE-250-TICK-LAST-INTERNAL"),
    trade_size=Decimal("0.10"),
    fast_ema_period=10,
    slow_ema_period=20,
    twap_horizon_secs=10.0,
    twap_interval_secs=2.5,
)

# Instantiate and add your strategy
strategy = EMACrossTWAP(config=strategy_config)
engine.add_strategy(strategy=strategy)

In [9]:
# Instantiate and add your execution algorithm
exec_algorithm = TWAPExecAlgorithm()  # Using defaults
engine.add_exec_algorithm(exec_algorithm)

In [10]:
# Run the engine (from start to end of data)
engine.run()

In [11]:
engine.trader.generate_account_report(BINANCE)

,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2020-08-14 10:00:00.223000+00:00,1000000.00000000,0.00000000,1000000.00000000,USDT,BINANCE-001,CASH,None,[],True,{}
2020-08-14 10:00:00.223000+00:00,10.00000000,0.00000000,10.00000000,ETH,BINANCE-001,CASH,None,[],True,{}
2020-08-14 10:22:34.574000+00:00,999989.38468857,0.00000000,999989.38468857,USDT,BINANCE-001,CASH,None,[],False,{}
2020-08-14 10:22:34.574000+00:00,10.02500000,0.00000000,10.02500000,ETH,BINANCE-001,CASH,None,[],False,{}
2020-08-14 10:22:37.074000+00:00,999978.76937714,0.00000000,999978.76937714,USDT,BINANCE-001,CASH,None,[],False,{}
...,...,...,...,...,...,...,...,...,...,...
2020-08-14 14:24:40.817000+00:00,10.07500000,0.00000000,10.07500000,ETH,BINANCE-001,CASH,None,[],False,{}
2020-08-14 14:24:43.317000+00:00,999956.20097365,0.00000000,999956.20097365,USDT,BINANCE-001,CASH,None,[],False,{}
2020-08-14 14:24:43.317000+00:00,10.10000000,0.00000000,10.10000000,ETH,BINANCE-001,CASH,None,[],False,{}
2020-08-14 14:59:58.693000+00:00,999998.88570475,0.00000000,999998.88570475,USDT,BINANCE-001,CASH,None,[],False,{}


In [12]:
engine.trader.generate_order_fills_report()

,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20200814-102234-001-000-1,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-004,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-004,MARKET,BUY,0.02500,...,None,None,None,TWAP,"{'horizon_secs': 10.0, 'interval_secs': 2.5}",O-20200814-102234-001-000-1,None,26635543-8279-4084-a3ac-7410d4352830,2020-08-14 10:22:34.574000+00:00,2020-08-14 10:22:42.074000+00:00
O-20200814-102234-001-000-1-E1,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-001,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-001,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-102234-001-000-1,None,fa50bec7-69f7-417a-b4b6-d898b4ff4e58,2020-08-14 10:22:34.574000+00:00,2020-08-14 10:22:34.574000+00:00
O-20200814-102234-001-000-1-E2,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-002,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-002,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-102234-001-000-1,None,befa370e-56a2-4776-a1f7-5277a33132e9,2020-08-14 10:22:37.074000+00:00,2020-08-14 10:22:37.074000+00:00
O-20200814-102234-001-000-1-E3,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-003,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-003,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-102234-001-000-1,None,c3ef7ba3-5c3c-4733-bd31-c50100a2cc59,2020-08-14 10:22:39.574000+00:00,2020-08-14 10:22:39.574000+00:00
O-20200814-103237-001-000-2,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-005,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-005,MARKET,SELL,0.10000,...,None,None,None,None,None,None,None,4d89ebb2-0436-42f7-a513-24b805ee9643,2020-08-14 10:32:37.428000+00:00,2020-08-14 10:32:37.428000+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O-20200814-142435-001-000-29,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-074,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-075,MARKET,BUY,0.02500,...,None,None,None,TWAP,"{'horizon_secs': 10.0, 'interval_secs': 2.5}",O-20200814-142435-001-000-29,None,e3c4f8d2-4406-4cdb-b482-afda3190b232,2020-08-14 14:24:35.817000+00:00,2020-08-14 14:24:43.317000+00:00
O-20200814-142435-001-000-29-E1,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-071,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-072,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-142435-001-000-29,None,4788415d-fce4-440d-955c-748e1e2b31bd,2020-08-14 14:24:35.817000+00:00,2020-08-14 14:24:35.817000+00:00
O-20200814-142435-001-000-29-E2,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-072,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-073,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-142435-001-000-29,None,fbec5e9f-5d2b-4ea4-8fb1-02e7d05ebe73,2020-08-14 14:24:38.317000+00:00,2020-08-14 14:24:38.317000+00:00


In [13]:
engine.trader.generate_positions_report()

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,...,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
ETHUSDT.BINANCE-EMACrossTWAP-000-77ca40b7-9bca-432c-9056-54aab5b0a593,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-102234-001-000-1-E1,O-20200814-103237-001-000-2,BUY,FLAT,0.00000,0.10000,...,2020-08-14 10:22:34.574000+00:00,1597401157428000000,2020-08-14 10:32:37.428000+00:00,602854000000,424.5625,423.490000,[0.00848054 USDT],-0.00253,-0.11573054 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-47b65ca9-484e-4c0f-addb-da66add90cd5,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-103237-001-000-3-E1,O-20200814-104611-001-000-4,SELL,FLAT,0.00000,0.10000,...,2020-08-14 10:32:37.428000+00:00,1597401971428000000,2020-08-14 10:46:11.428000+00:00,814000000000,423.4875,424.700000,[0.00848189 USDT],-0.00286,-0.12973189 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-f77f009f-450c-4dbd-8197-58448dcb8913,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-104611-001-000-5-E1,O-20200814-110002-001-000-6,BUY,FLAT,0.00000,0.10000,...,2020-08-14 10:46:11.428000+00:00,1597402802097000000,2020-08-14 11:00:02.097000+00:00,830669000000,424.6050,424.143982,[0.00848750 USDT],-0.00109,-0.05458930 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-ae9db08d-04b6-4e1a-a10d-41abbbd30c7f,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-110002-001-000-7-E1,O-20200814-111402-001-000-8,SELL,FLAT,0.00000,0.10000,...,2020-08-14 11:00:02.097000+00:00,1597403642429000000,2020-08-14 11:14:02.429000+00:00,840332000000,424.0050,425.570000,[0.00849575 USDT],-0.00369,-0.16499575 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-aa1e521d-f5af-41fb-b6ea-c6de6a45514a,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-111402-001-000-9-E1,O-20200814-113300-001-000-10,BUY,FLAT,0.00000,0.10000,...,2020-08-14 11:14:02.429000+00:00,1597404780084000000,2020-08-14 11:33:00.084000+00:00,1137655000000,425.6375,425.340000,[0.00850979 USDT],-0.00070,-0.03825979 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-49512960-1429-4def-b330-1c63cd52c374,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-113300-001-000-11-E1,O-20200814-113705-001-000-12,SELL,FLAT,0.00000,0.10000,...,2020-08-14 11:33:00.084000+00:00,1597405025204000000,2020-08-14 11:37:05.204000+00:00,245120000000,425.4350,426.250000,[0.00851685 USDT],-0.00192,-0.09001685 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-aa236cf7-bf51-4db6-bd21-d14350194bca,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-113705-001-000-13-E1,O-20200814-120215-001-000-14,BUY,FLAT,0.00000,0.07500,...,2020-08-14 11:37:05.204000+00:00,1597406535235000000,2020-08-14 12:02:15.235000+00:00,1510031000000,426.3100,426.430000,[0.00639556 USDT],0.00028,0.00260444 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-7d03208e-7bc7-4091-8add-d72926ae246f,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-120215-001-000-15-E1,O-20200814-123931-001-000-16,SELL,FLAT,0.00000,0.10000,...,2020-08-14 12:02:15.235000+00:00,1597408771933000000,2020-08-14 12:39:31.933000+00:00,2236698000000,426.3600,427.330000,[0.00853691 USDT],-0.00228,-0.10553691 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-146f64bd-29af-4b76-9685-59c99c9d3228,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-123931-001-000-17-E1,O-20200814-125346-001-000-18,BUY,FLAT,0.00000,0.10000,...,2020-08-14 12:39:31.933000+00:00,1597409626057000000,2020-08-14 12:53:46.057000+00:00,854124000000,427.3225,426.260000,[0.00853583 USDT],-0.00249,-0.11478583 USDT,True


In [14]:
# For repeated backtest runs, reset the engine
engine.reset()

# Instruments and data persist, just add new components and run again

In [15]:
# Once done, good practice to dispose of the object if the script continues
engine.dispose()

In [17]:
df = pd.read_parquet(r"D:\nautilus_trader\nautilus_trader\OHLCV_40.par")
df.index , df.columns 

(Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
        ...
         5,  6,  7,  8,  9, 10,  0,  1,  2,  3],
       dtype='int64', length=36380),
 Index(['id', 'symbol', 'ts', 'open', 'high', 'low', 'close', 'volume',
        'market_cap'],
       dtype='object'))

In [18]:
df.ts

0    2025-11-27
1    2025-11-28
2    2025-11-29
3    2025-11-30
4    2025-12-01
        ...    
10   2026-03-10
0    2026-03-07
1    2026-03-08
2    2026-03-09
3    2026-03-10
Name: ts, Length: 36380, dtype: datetime64[ns]

In [19]:
df

,id,symbol,ts,open,high,low,close,volume,market_cap
0,39001,ZOOTOPIA,2025-11-27,3.053700e-14,3.582500e-14,2.789900e-14,2.907700e-14,6744.25,0.00
1,39001,ZOOTOPIA,2025-11-28,2.907700e-14,2.951900e-14,2.369800e-14,2.951900e-14,3684.81,0.00
2,39001,ZOOTOPIA,2025-11-29,2.951900e-14,3.001100e-14,2.642700e-14,2.827900e-14,1125.76,0.00
3,39001,ZOOTOPIA,2025-11-30,2.827900e-14,2.827900e-14,2.709500e-14,2.716300e-14,353.70,0.00
4,39001,ZOOTOPIA,2025-12-01,2.716300e-14,2.716300e-14,2.188100e-14,2.512700e-14,1090.99,0.00
...,...,...,...,...,...,...,...,...,...
10,39671,龙虾,2026-03-10,1.354624e-02,1.959129e-02,1.025590e-02,1.201996e-02,34346570.49,12019962.36
0,39679,GF,2026-03-07,4.189798e-03,8.246284e-03,2.682533e-03,3.328923e-03,1578877.58,0.00
1,39679,GF,2026-03-08,3.319557e-03,3.484163e-03,2.524968e-03,2.627216e-03,773128.69,0.00
2,39679,GF,2026-03-09,2.626621e-03,6.862163e-03,2.559917e-03,5.550545e-03,1204297.40,0.00


In [20]:
from decimal import Decimal
import pandas as pd
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.examples.strategies.ema_cross import EMACross, EMACrossConfig
from nautilus_trader.model import BarType, Money, TraderId, Venue
from nautilus_trader.model.currencies import BTC, USDT
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.test_kit.providers import TestInstrumentProvider

In [22]:
df = pd.read_parquet(r"D:\nautilus_trader\nautilus_trader\OHLCV_40.par")

# Convert 'ts' to datetime UTC and set as index
df["timestamp"] = pd.to_datetime(df["ts"], utc=True)
df = df.set_index("timestamp")

# Keep only OHLCV columns (what BarDataWrangler expects)
df = df[["open", "high", "low", "close", "volume"]]

# Sort by timestamp
df = df.sort_index()
df.head()

,open,high,low,close,volume
timestamp,,,,,
2022-08-02 00:00:00+00:00,9.327033e-10,9.327033e-10,9.327033e-10,9.327033e-10,75989.01
2022-08-03 00:00:00+00:00,9.327033e-10,9.327033e-10,9.327033e-10,9.327033e-10,75989.01
2022-08-04 00:00:00+00:00,9.327033e-10,9.327033e-10,9.327033e-10,9.327033e-10,75989.01
2022-08-05 00:00:00+00:00,9.327033e-10,9.327033e-10,9.327033e-10,9.327033e-10,75989.01
2022-08-06 00:00:00+00:00,9.327033e-10,9.327033e-10,9.327033e-10,9.327033e-10,75989.01


In [23]:
df.tail()

,open,high,low,close,volume
timestamp,,,,,
2026-03-10 00:00:00+00:00,0.005551,0.005920,0.004293,0.004543,683249.33
2026-03-11 00:00:00+00:00,0.042384,0.042913,0.013501,0.042123,4909.93
2026-03-11 00:00:00+00:00,52.266630,52.813833,50.827831,51.276800,0.00
2026-03-12 00:00:00+00:00,0.042123,0.042123,0.030515,0.040930,265.96
2026-03-12 00:00:00+00:00,51.276800,51.345595,49.800547,50.877182,0.00


In [24]:
BTCUSDT_BINANCE = TestInstrumentProvider.btcusdt_binance()

# CRITICAL: Use BarDataWrangler (not TradeTickDataWrangler)
# CRITICAL: BarType must end with "EXTERNAL" for pre-aggregated bar data
bar_type = BarType.from_str("BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL")

wrangler = BarDataWrangler(bar_type=bar_type, instrument=BTCUSDT_BINANCE)
bars = wrangler.process(df)
bars[:3]

[Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,0.00,0.00,0.00,0.00,75989.010000,1659398400000000000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,0.00,0.00,0.00,0.00,75989.010000,1659484800000000000),
 Bar(BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL,0.00,0.00,0.00,0.00,75989.010000,1659571200000000000)]

In [25]:
config = BacktestEngineConfig(trader_id=TraderId("BACKTESTER-001"))
engine = BacktestEngine(config=config)

BINANCE = Venue("BINANCE")
engine.add_venue(
    venue=BINANCE,
    oms_type=OmsType.NETTING,
    account_type=AccountType.CASH,
    base_currency=None,  # Multi-currency
    starting_balances=[Money(1_000_000.0, USDT), Money(10.0, BTC)],
)

engine.add_instrument(BTCUSDT_BINANCE)
engine.add_data(bars)

In [26]:
# EMACross (simpler than EMACrossTWAP - no execution algorithm needed)
strategy_config = EMACrossConfig(
    instrument_id=BTCUSDT_BINANCE.id,
    bar_type=bar_type,
    trade_size=Decimal("0.01"),
    fast_ema_period=10,
    slow_ema_period=20,
)
strategy = EMACross(config=strategy_config)
engine.add_strategy(strategy=strategy)

engine.run()

In [27]:
engine.trader.generate_account_report(BINANCE)


,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2022-08-02 00:00:00+00:00,1000000.00000000,0.00000000,1000000.00000000,USDT,BINANCE-001,CASH,None,[],True,{}
2022-08-02 00:00:00+00:00,10.00000000,0.00000000,10.00000000,BTC,BINANCE-001,CASH,None,[],True,{}
2024-10-25 00:00:00+00:00,999999.05075170,0.00000000,999999.05075170,USDT,BINANCE-001,CASH,None,[],False,{}
2024-10-25 00:00:00+00:00,10.01000000,0.00000000,10.01000000,BTC,BINANCE-001,CASH,None,[],False,{}
2024-11-23 00:00:00+00:00,1000001.05894150,0.00000000,1000001.05894150,USDT,BINANCE-001,CASH,None,[],False,{}
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 00:00:00+00:00,9.99000000,0.00000000,9.99000000,BTC,BINANCE-001,CASH,None,[],False,{}
2025-12-31 00:00:00+00:00,969502.52562715,0.00000000,969502.52562715,USDT,BINANCE-001,CASH,None,[],False,{}
2025-12-31 00:00:00+00:00,9.99000100,0.00000000,9.99000100,BTC,BINANCE-001,CASH,None,[],False,{}
2025-12-31 00:00:00+00:00,968277.57904084,0.00000000,968277.57904084,USDT,BINANCE-001,CASH,None,[],False,{}


In [28]:
engine.trader.generate_order_fills_report()

,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20241025-000000-001-000-1,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-001,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-818,MARKET,BUY,0.010000,...,None,None,None,None,None,None,None,9a3d056b-a461-4bd4-8fbd-2c245ba773d7,2024-10-25 00:00:00+00:00,2024-10-25 00:00:00+00:00
O-20241123-000000-001-000-2,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-002,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-878,MARKET,SELL,0.010000,...,None,None,None,None,None,None,None,2f01bc46-c861-4aa3-838e-3d7aabd74bb1,2024-11-23 00:00:00+00:00,2024-11-23 00:00:00+00:00
O-20241123-000000-001-000-3,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-003,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-879,MARKET,SELL,0.010000,...,None,None,None,None,None,None,None,046c86e5-17e9-4da9-a719-72ea80944fa9,2024-11-23 00:00:00+00:00,2024-11-23 00:00:00+00:00
O-20241124-000000-001-000-4,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-004,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-881,MARKET,BUY,0.010000,...,None,None,None,None,None,None,None,32b40ca3-9bd5-47f3-bd2b-44f27aa9fd5b,2024-11-24 00:00:00+00:00,2024-11-24 00:00:00+00:00
O-20241124-000000-001-000-5,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-005,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-882,MARKET,BUY,0.010000,...,None,None,None,None,None,None,None,4b5d2093-01c7-43f2-9db4-2fc536a4646a,2024-11-24 00:00:00+00:00,2024-11-24 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O-20251231-000000-001-000-1065,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-1065,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-10708,MARKET,BUY,0.010000,...,None,None,None,None,None,None,None,a2b08ff2-ebd7-4ef9-95f8-71ca298c422e,2025-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00
O-20251231-000000-001-000-1066,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-1066,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-10712,MARKET,SELL,0.010000,...,None,None,None,None,None,None,None,6533319c-8404-459c-9738-0345dcb97f47,2025-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00
O-20251231-000000-001-000-1067,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-1-1067,BTCUSDT.BINANCE-EMACross-000,BINANCE-001,BINANCE-1-10713,MARKET,SELL,0.010000,...,None,None,None,None,None,None,None,8edff5b7-81c5-4dee-858c-5810e139f567,2025-12-31 00:00:00+00:00,2025-12-31 00:00:00+00:00


In [29]:

engine.trader.generate_positions_report()

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,...,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
BTCUSDT.BINANCE-EMACross-000-b98d46b9-06b5-4e8b-a197-1f99ade6bb37,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20241025-000000-001-000-1,O-20241123-000000-001-000-2,BUY,FLAT,0.000000,0.010000,...,2024-10-25 00:00:00+00:00,1732320000000000000,2024-11-23 00:00:00+00:00,2.505600e+15,94.83,2.010200e+02,[0.00295850 USDT],1.119790e+00,1.05894150 USDT,True
BTCUSDT.BINANCE-EMACross-000-b6b14530-30ad-4dfb-8c23-cec8fef536fd,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20241123-000000-001-000-3,O-20241124-000000-001-000-4,SELL,FLAT,0.000000,0.010000,...,2024-11-23 00:00:00+00:00,1732406400000000000,2024-11-24 00:00:00+00:00,8.640000e+13,201.02,2.008400e+02,[0.00401860 USDT],9.000000e-04,-0.00221860 USDT,True
BTCUSDT.BINANCE-EMACross-000-b1b9c1e1-f582-46b6-90ab-d57f6a87ae24,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20241124-000000-001-000-5,O-20241125-000000-001-000-6,BUY,FLAT,0.000000,0.010000,...,2024-11-24 00:00:00+00:00,1732492800000000000,2024-11-25 00:00:00+00:00,8.640000e+13,200.84,2.015500e+02,[0.00402390 USDT],3.540000e-03,0.00307610 USDT,True
BTCUSDT.BINANCE-EMACross-000-11e665aa-0573-43d8-8e67-c25d13f1e231,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20241125-000000-001-000-7,O-20241127-000000-001-000-8,SELL,FLAT,0.000000,0.010000,...,2024-11-25 00:00:00+00:00,1732665600000000000,2024-11-27 00:00:00+00:00,1.728000e+14,201.55,2.020300e+02,[0.00403580 USDT],-2.380000e-03,-0.00883580 USDT,True
BTCUSDT.BINANCE-EMACross-000-80255e02-db18-42a4-a7a2-1cadff7d4d8a,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20241127-000000-001-000-9,O-20241218-000000-001-000-10,BUY,FLAT,0.000000,0.010000,...,2024-11-27 00:00:00+00:00,1734480000000000000,2024-12-18 00:00:00+00:00,1.814400e+15,202.03,1.360400e+02,[0.00338070 USDT],-3.266300e-01,-0.66328070 USDT,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
BTCUSDT.BINANCE-EMACross-000-91197286-d733-4514-bfe5-7875e6e7deef,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20251230-000000-001-000-1053,O-20251230-000000-001-000-1054,BUY,FLAT,0.000000,0.010000,...,2025-12-30 00:00:00+00:00,1767052800000000000,2025-12-30 00:00:00+00:00,NaN,88530.89,1.300000e-01,[0.88531020 USDT],-1.000000e+00,-886.19291020 USDT,True
BTCUSDT.BINANCE-EMACross-000-f9910317-5833-4403-ab0c-7471fed3b1f3,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20251230-000000-001-000-1061,O-20251230-000000-001-000-1062,BUY,FLAT,0.000000,0.010000,...,2025-12-30 00:00:00+00:00,1767052800000000000,2025-12-30 00:00:00+00:00,NaN,285.99,6.000000e-02,[0.00286050 USDT],-9.997900e-01,-2.86216050 USDT,True
BTCUSDT.BINANCE-EMACross-000-5292d4f5-ab2a-4fa5-989c-8c143cf0c20a,BACKTESTER-001,EMACross-000,BTCUSDT.BINANCE,BINANCE-001,O-20251230-000000-001-000-1063,O-20251231-000000-001-000-1064,SELL,FLAT,0.000000,0.010000,...,2025-12-30 00:00:00+00:00,1767139200000000000,2025-12-31 00:00:00+00:00,8.640000e+13,0.06,2.566000e+01,[0.00025720 USDT],-4.266667e+02,-0.25625720 USDT,True


In [30]:
from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.backtest.engine import BacktestEngine

# After running your backtest
engine.run()

# Generate tearsheet
create_tearsheet(
    engine=engine,
    output_path="backtest_results_custom_file.html",
)

In [31]:
engine.reset()
engine.dispose()

In [32]:
from nautilus_trader.analysis import TearsheetConfig
from nautilus_trader.analysis import TearsheetDrawdownChart
from nautilus_trader.analysis import TearsheetEquityChart
from nautilus_trader.analysis import TearsheetRunInfoChart
from nautilus_trader.analysis import TearsheetStatsTableChart

config = TearsheetConfig(
    charts=[
        TearsheetRunInfoChart(),
        TearsheetStatsTableChart(),
        TearsheetEquityChart(),
        TearsheetDrawdownChart(),
    ],
    theme="nautilus_dark",
    height=2000,
)

create_tearsheet(
    engine=engine,
    output_path="custom_tearsheet.html",
    config=config,
)

In [33]:
from nautilus_trader.analysis import TearsheetConfig
from nautilus_trader.analysis import TearsheetBarsWithFillsChart
from nautilus_trader.analysis import TearsheetDistributionChart
from nautilus_trader.analysis import TearsheetDrawdownChart
from nautilus_trader.analysis import TearsheetEquityChart
from nautilus_trader.analysis import TearsheetMonthlyReturnsChart
from nautilus_trader.analysis import TearsheetRollingSharpeChart
from nautilus_trader.analysis import TearsheetRunInfoChart
from nautilus_trader.analysis import TearsheetStatsTableChart
from nautilus_trader.analysis import TearsheetYearlyReturnsChart

config = TearsheetConfig(
    charts=[
        TearsheetRunInfoChart(),
        TearsheetStatsTableChart(),
        TearsheetEquityChart(),
        TearsheetDrawdownChart(),
        TearsheetMonthlyReturnsChart(),
        TearsheetDistributionChart(),
        TearsheetRollingSharpeChart(),
        TearsheetYearlyReturnsChart(),
        TearsheetBarsWithFillsChart(bar_type="BTCUSDT.BINANCE-1-DAY-LAST-EXTERNAL"),
    ],
    theme="nautilus_dark",
    height=2000,
)

create_tearsheet(
    engine=engine,
    output_path="custom_tearsheet_2.html",
    config=config,
)
